In [14]:
import re
import glob
import pandas as pd
from pathlib import Path
from datetime import datetime

## Training

In [15]:
# Constants
MODELS_ROOT = "./models"
TIME_FORMAT = "%Y-%m-%d %H:%M:%S"
TIME_PATTERN = r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})"
PHASE2_START_PATTERN = r"Calibration Epoch 01/\d+ \| Step 0*0\b"
PHASE2_END_PATTERN = r"Decoder calibration complete"

SELECTED_EXPERIMENTS = {
    # COCO
    "vq_coco_16x16_k1024_d64",
    "vqema_coco_16x16_k1024_d64",
    "fsq_coco_16x16_k1000_d64",
    "diveq_coco_16x16_k1024_d64",
    "stgs_coco_16x16_k1024_d64",
    "avq_coco_16x16_k1024_d64",
    "rvqema_coco_16x16_q3_k1024_d64",
    "rdiveq_coco_16x16_q3_k1024_d64",
    "rstgs_coco_16x16_q3_k1024_d64",
    "arvq_coco_16x16_q3_k1024_d64",

    # MiniPlaces
    "vq_32x32_k1024_d64",
    "vqema_32x32_k1024_d64",
    "fsq_32x32_k1000_d64",
    "diveq_32x32_k1024_d64",
    "stgs_32x32_k1024_d64",
    "avq_32x32_k1024_d64",
    "rvqema_32x32_q3_k1024_d64",
    "rdiveq_32x32_q3_k1024_d64",
    "rstgs_32x32_q3_k1024_d64",
    "arvq_32x32_q3_k1024_d64",

    # Tiny ImageNet
    "vq_16x16_k1024_d64",
    "vqema_16x16_k1024_d64",
    "fsq_16x16_k1000_d64",
    "diveq_16x16_k1024_d64",
    "stgs_16x16_k1024_d64",
    "avq_16x16_k1024_d64",
    "rvqema_16x16_q3_k1024_d64",
    "rdiveq_16x16_q3_k1024_d64",
    "rstgs_16x16_q3_k1024_d64",
    "arvq_16x16_q3_k1024_d64",
    
}

In [16]:
def get_regex(pattern):
    return re.compile(pattern)

def parse_timestamp(line):
    regex = get_regex(TIME_PATTERN)
    match = regex.match(line)
    if not match:
        return None
    return datetime.strptime(match.group(1), TIME_FORMAT)

def hours(duration):
    return duration.total_seconds() / 3600.0

def minutes(duration):
    return duration.total_seconds() / 60.0

def parse_log(log_path):
    """
    o/p: dict w/ start/end timestamps and phase boundaries
    """
    start_time = None
    end_time = None
    phase2_start_time = None
    phase2_end_time = None
    phase2_start_regex = get_regex(PHASE2_START_PATTERN)
    phase2_end_regex = get_regex(PHASE2_END_PATTERN)
    with open(log_path, 'r') as f:
        for line in f:
            timestamp = parse_timestamp(line)
            if timestamp is None:
                continue
            if start_time is None:
                start_time = timestamp
            end_time = timestamp # keep advancing to last valid timestamp

            if phase2_start_time is None and phase2_start_regex.search(line):
                phase2_start_time = timestamp
            if phase2_end_regex.search(line):
                phase2_end_time = timestamp

    return {
        "start": start_time,
        "end": end_time,
        "phase2_start": phase2_start_time,
        "phase2_end": phase2_end_time,
    }

def process_logs():
    root_dir = Path(MODELS_ROOT)
    log_paths = [
        p for p in root_dir.glob("*/*/*/train.log") if p.parent.name in SELECTED_EXPERIMENTS
    ]

    output = []
    for log_path in log_paths:
        experiment = log_path.parent.name
        method = experiment.split('_')[0].upper()
        seed = int(log_path.parent.parent.name.split('-')[-1])
        dataset = log_path.parent.parent.parent.name
        data = parse_log(log_path)

        start = data["start"]
        end = data["end"]
        phase2_start = data["phase2_start"]
        phase2_end = data["phase2_end"]

        total_duration = (end - start) if (start and end) else None
        phase1_duration = (phase2_start - start) if (start and phase2_start) else None
        phase2_duration = (phase2_end - phase2_start) if (phase2_start and phase2_end) else None
        posthoc_duration = (end - phase2_end) if (end and phase2_end) else None

        output.append({
            "experiment": experiment,
            "method": method,
            "seed": seed,
            "dataset": dataset,
            "start": start,
            "end": end,
            "total_hours": hours(total_duration),
            "total_minutes": minutes(total_duration),
            "phase1_hours": hours(phase1_duration),
            "phase1_minutes": minutes(phase1_duration),
            "phase2_hours": hours(phase2_duration),
            "phase2_minutes": minutes(phase2_duration),
            "posthoc_minutes": minutes(posthoc_duration),
        })

    return output

def summarize_results(df, metrics=None):
    """
    Compute mean and sample standard deviation across model seeds.
    """
    summary = df.groupby(["dataset", "experiment"], sort=False)[metrics].agg(["mean", "std"])
    summary.columns = ["_".join(col) for col in summary.columns]
    return summary.reset_index()

In [17]:
output = process_logs()

In [18]:
df = pd.DataFrame(output)
df.head(3)

,experiment,method,seed,dataset,start,end,total_hours,total_minutes,phase1_hours,phase1_minutes,phase2_hours,phase2_minutes,posthoc_minutes
0,arvq_coco_16x16_q3_k1024_d64,ARVQ,2027,coco,2026-09-16 06:23:49,2026-09-16 12:29:16,6.090833,365.450000,5.692778,341.566667,0.216111,12.966667,10.916667
1,avq_coco_16x16_k1024_d64,AVQ,2027,coco,2026-09-16 01:09:17,2026-09-16 06:19:41,5.173333,310.400000,4.478611,268.716667,0.512778,30.766667,10.916667
2,diveq_coco_16x16_k1024_d64,DIVEQ,2027,coco,2026-10-02 22:15:52,2026-10-03 02:40:35,4.411944,264.716667,4.025556,241.533333,0.205556,12.333333,10.850000


In [19]:
metric_cols = [
    "total_hours",
    "phase1_hours",
    "phase2_minutes",
    "posthoc_minutes",
]

summary = summarize_results(df, metrics=metric_cols)
summary["method"] = summary["experiment"].apply(lambda row: row.split('_')[0].upper())
del summary["experiment"]
summary

,dataset,total_hours_mean,total_hours_std,phase1_hours_mean,phase1_hours_std,phase2_minutes_mean,phase2_minutes_std,posthoc_minutes_mean,posthoc_minutes_std,method
0,coco,6.123426,0.054539,5.685185,0.015116,15.450000,4.286834,10.844444,0.110972,ARVQ
1,coco,5.161944,0.013002,4.468056,0.011314,30.761111,0.041944,10.872222,0.076980,AVQ
2,coco,4.495648,0.072543,4.007685,0.015749,18.427778,5.277950,10.850000,0.016667,DIVEQ
3,coco,4.541019,0.070891,3.892778,0.009639,28.055556,3.555525,10.838889,0.120570,FSQ
4,coco,4.731111,0.063838,4.294259,0.012289,15.327778,3.829902,10.883333,0.000000,RDIVEQ
5,coco,6.688611,0.651684,6.293796,0.651765,12.444444,0.009623,11.244444,0.019245,RSTGS
6,coco,4.604167,0.003643,4.206204,0.003047,12.861111,0.009623,11.016667,0.044096,RVQEMA
7,coco,5.006204,0.049449,4.567870,0.001283,15.138889,3.041670,11.161111,0.009623,STGS
8,coco,4.333981,0.046624,3.915185,0.010185,14.300000,3.507413,10.827778,0.110972,VQ
9,coco,4.561852,0.160700,3.971296,0.018655,24.522222,10.599454,10.911111,0.125093,VQEMA


In [20]:
print(summary.to_latex(index=False))

\begin{tabular}{lrrrrrrrrl}
\toprule
dataset & total_hours_mean & total_hours_std & phase1_hours_mean & phase1_hours_std & phase2_minutes_mean & phase2_minutes_std & posthoc_minutes_mean & posthoc_minutes_std & method \\
\midrule
coco & 6.123426 & 0.054539 & 5.685185 & 0.015116 & 15.450000 & 4.286834 & 10.844444 & 0.110972 & ARVQ \\
coco & 5.161944 & 0.013002 & 4.468056 & 0.011314 & 30.761111 & 0.041944 & 10.872222 & 0.076980 & AVQ \\
coco & 4.495648 & 0.072543 & 4.007685 & 0.015749 & 18.427778 & 5.277950 & 10.850000 & 0.016667 & DIVEQ \\
coco & 4.541019 & 0.070891 & 3.892778 & 0.009639 & 28.055556 & 3.555525 & 10.838889 & 0.120570 & FSQ \\
coco & 4.731111 & 0.063838 & 4.294259 & 0.012289 & 15.327778 & 3.829902 & 10.883333 & 0.000000 & RDIVEQ \\
coco & 6.688611 & 0.651684 & 6.293796 & 0.651765 & 12.444444 & 0.009623 & 11.244444 & 0.019245 & RSTGS \\
coco & 4.604167 & 0.003643 & 4.206204 & 0.003047 & 12.861111 & 0.009623 & 11.016667 & 0.044096 & RVQEMA \\
coco & 5.006204 & 0.049449 & 4.

## Inference

In [21]:
import json

In [22]:
MODELS_ROOT = Path("./models")

path_16x16 = MODELS_ROOT / "benchmark_16x16.json"
path_32x32 = MODELS_ROOT / "benchmark_32x32.json"

In [23]:
METHOD_ORDER = [
    "vq", "vqema", "fsq", "diveq", "stgs", "avq", "rvqema", "rdiveq", "rstgs", "arvq",
]

In [24]:
def load_benchmark(path):
    with open(path, 'r') as f:
        data = json.load(f)

    rows = {}
    batch_size = data["batch_size"]
    for method in METHOD_ORDER:
        row = data["methods"][method]

        # Convert batch level timing to per image timing
        rows[method.upper()] = {
            "latency_mean": row["mean_ms_per_image"],
            "latency_std": row["std_batch_ms"] / row["batch_size"],
            "throughput": row["images_per_second"],
            "workspace": row["peak_workspace_mb"],
        }

    metadata = {
        "gpu": data["gpu"],
        "torch_version": data["torch_version"],
        "batch_size": batch_size,
        "embedding_dim": data["embedding_dim"],
        "latent_grid": data["latent_grid"],
        "temperature": data["temperature"],
        "warmup": data["warmup"],
        "repeats": data["repeats"],
    }

    return rows, metadata

In [25]:
rows16x16, meta16x16 = load_benchmark(path_16x16)
rows32x32, meta32x32 = load_benchmark(path_32x32)

df16 = pd.DataFrame(rows16x16).T
df32 = pd.DataFrame(rows32x32).T

df16["latent_grid"] = "16x16"
df32["latent_grid"] = "32x32"

df = pd.concat([df16, df32], axis=0).reset_index(names="method")
df

,method,latency_mean,latency_std,throughput,workspace,latent_grid
0,VQ,0.037094,0.001828,26958.872205,200.000000,16x16
1,VQEMA,0.036394,0.001480,27477.347382,200.000000,16x16
2,FSQ,0.006121,0.003040,163374.399912,26.125000,16x16
3,DIVEQ,0.055428,0.001389,18041.297712,200.000000,16x16
4,STGS,0.022854,0.002438,43756.696119,164.125488,16x16
5,AVQ,0.041637,0.001400,24017.333780,204.250000,16x16
6,RVQEMA,0.113941,0.000286,8776.472818,232.250977,16x16
7,RDIVEQ,0.166397,0.001479,6009.716318,216.250000,16x16
8,RSTGS,0.070431,0.003040,14198.357936,316.375488,16x16
9,ARVQ,0.127256,0.002086,7858.205309,420.500000,16x16


In [26]:
with open(path_16x16, 'r') as f:
    content = json.load(f)

content

{'gpu': 'NVIDIA RTX 2000 Ada Generation',
 'torch_version': '2.8.0+cu128',
 'batch_size': 64,
 'embedding_dim': 64,
 'latent_grid': 16,
 'temperature': 0.02,
 'warmup': 20,
 'repeats': 100,
 'methods': {'vq': {'method': 'vq',
   'batch_size': 64,
   'latent_shape': [64, 16, 16],
   'warmup_runs': 20,
   'timed_runs': 100,
   'mean_batch_ms': 2.3739865493774412,
   'std_batch_ms': 0.11698903058618484,
   'median_batch_ms': 2.36135995388031,
   'mean_ms_per_image': 0.03709353983402252,
   'images_per_second': 26958.872204555446,
   'peak_workspace_mb': 200.0,
   'timings_ms': [3.5272319316864014,
    2.3722879886627197,
    2.3663039207458496,
    2.3778560161590576,
    2.359391927719116,
    2.371999979019165,
    2.360032081604004,
    2.368896007537842,
    2.358880043029785,
    2.3695359230041504,
    2.355135917663574,
    2.3690879344940186,
    2.3559999465942383,
    2.364192008972168,
    2.3568320274353027,
    2.37174391746521,
    2.3583359718322754,
    2.372512102127075,


In [27]:
meta16x16

{'gpu': 'NVIDIA RTX 2000 Ada Generation',
 'torch_version': '2.8.0+cu128',
 'batch_size': 64,
 'embedding_dim': 64,
 'latent_grid': 16,
 'temperature': 0.02,
 'warmup': 20,
 'repeats': 100}

In [28]:
meta32x32

{'gpu': 'NVIDIA RTX 2000 Ada Generation',
 'torch_version': '2.8.0+cu128',
 'batch_size': 16,
 'embedding_dim': 64,
 'latent_grid': 32,
 'temperature': 0.02,
 'warmup': 20,
 'repeats': 100}